# Day 3 – Data Quality Assessment

## Project
Sales & Business Performance Analytics Dashboard

## Objective
Assess the raw Superstore dataset for duplicate records, row grain, numeric anomalies, date validity, and categorical consistency.

**Scope:** This notebook performs assessment only. It does not clean, transform, overwrite, or export the raw dataset.

## 1. Load the raw dataset

In [1]:
import pandas as pd

DATA_PATH = "../../data/raw/sample_-_superstore.xls"

df = pd.read_excel(DATA_PATH, engine="xlrd")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (10194, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,US-2023-103800,2023-01-03,2023-01-07,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.2,5.5512
1,2,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870
2,3,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717
3,4,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748
4,5,US-2023-141817,2023-01-05,2023-01-12,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,...,19143,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536,3,0.2,4.8840


## 2. Duplicate and row-grain assessment

In [2]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate Row IDs:", df["Row ID"].duplicated().sum())
print("Unique Row IDs:", df["Row ID"].nunique())
print("Unique Order IDs:", df["Order ID"].nunique())

order_counts = df["Order ID"].value_counts()
print("Orders appearing more than once:", (order_counts > 1).sum())
print("Maximum rows per order:", order_counts.max())

display(order_counts.value_counts().sort_index().rename_axis("Lines per order").to_frame("Number of orders"))

Exact duplicate rows: 0
Duplicate Row IDs: 0
Unique Row IDs: 10194
Unique Order IDs: 5111
Orders appearing more than once: 2518
Maximum rows per order: 14


,Number of orders
Lines per order,
1,2593
2,1242
3,612
4,344
5,164
6,72
7,51
8,16
9,10


### Interpretation

Each row has a unique Row ID, while many Order IDs occur on multiple rows. This supports treating the dataset as order-line-level data rather than one row per order. Repeated Order IDs are not, by themselves, duplicate records.

## 3. Numeric value assessment

In [3]:
numeric_columns = ["Quantity", "Discount", "Sales", "Profit"]

numeric_summary = df[numeric_columns].agg(["min", "max"]).T
numeric_summary["zero_count"] = (df[numeric_columns] == 0).sum()
numeric_summary["negative_count"] = (df[numeric_columns] < 0).sum()

display(numeric_summary)

print("Quantity <= 0:", (df["Quantity"] <= 0).sum())
print("Discount outside 0 to 1:", (~df["Discount"].between(0, 1)).sum())

,min,max,zero_count,negative_count
Quantity,1.000,14.000,0,0
Discount,0.000,0.800,4925,0
Sales,0.444,22638.480,0,0
Profit,-6599.978,8399.976,65,1901


Quantity <= 0: 0
Discount outside 0 to 1: 0


### Interpretation

Quantity values are positive, discounts fall between 0 and 0.8, and sales values are positive. Negative Profit values may represent genuine losses on transactions and should not be removed without business justification.

## 4. Date validity assessment

In [4]:
order_dates = pd.to_datetime(df["Order Date"], errors="coerce")
ship_dates = pd.to_datetime(df["Ship Date"], errors="coerce")

print("Invalid Order Dates:", order_dates.isna().sum())
print("Invalid Ship Dates:", ship_dates.isna().sum())
print("Ship Date before Order Date:", (ship_dates < order_dates).sum())

print("Order Date range:", order_dates.min(), "to", order_dates.max())
print("Ship Date range:", ship_dates.min(), "to", ship_dates.max())

Invalid Order Dates: 0
Invalid Ship Dates: 0
Ship Date before Order Date: 0
Order Date range: 2023-01-03 00:00:00 to 2026-12-30 00:00:00
Ship Date range: 2023-01-07 00:00:00 to 2027-01-05 00:00:00


## 5. Categorical consistency assessment

In [5]:
categorical_columns = [
    "Ship Mode", "Segment", "Country/Region",
    "Region", "Category", "Sub-Category"
]

for column in categorical_columns:
    values = df[column].astype("string")
    whitespace_count = values.ne(values.str.strip()).sum()

    print(f"\\n{column}")
    print("Unique values:", df[column].nunique())
    print("Leading/trailing whitespace:", whitespace_count)
    display(df[column].value_counts(dropna=False).to_frame("Count"))

\nShip Mode
Unique values: 4
Leading/trailing whitespace: 0


,Count
Ship Mode,
Standard Class,6120
Second Class,1979
First Class,1548
Same Day,547


\nSegment
Unique values: 3
Leading/trailing whitespace: 0


,Count
Segment,
Consumer,5281
Corporate,3090
Home Office,1823


\nCountry/Region
Unique values: 2
Leading/trailing whitespace: 0


,Count
Country/Region,
United States,9994
Canada,200


... (2 outputs left out)

\nCategory
Unique values: 3
Leading/trailing whitespace: 0


,Count
Category,
Office Supplies,6128
Furniture,2201
Technology,1865


\nSub-Category
Unique values: 17
Leading/trailing whitespace: 0


,Count
Sub-Category,
Binders,1548
Paper,1384
Furnishings,1009
Phones,903
Storage,856
Art,821
Accessories,775
Chairs,634
Appliances,474


## 6. Day 3 findings

- No exact duplicate rows or duplicate Row IDs were identified.
- Repeated Order IDs indicate multiple product lines within orders.
- Quantity, discount, and sales checks found no invalid values under the checks performed.
- Negative Profit values require business interpretation, not automatic removal.
- No invalid dates or ship dates preceding order dates were found.
- The six inspected categorical columns contain no leading or trailing whitespace.

## Conclusion

The assessment did not identify obvious structural, numeric, date, or label-format errors in the checks performed. Negative profit values are retained as potentially valid business outcomes. The raw dataset remains unchanged.

**Next step:** Use these findings to guide the later cleaning stage; do not treat this assessment as proof that every field is error-free.